## Process - phase 2a, AAH/AEEH (no qf-batch)
- Load the cleaned beneficiary rows left by `clean_msa_1_before_qf_batch.ipynb`
- Select the AAH and AEEH eligibility routes
- Output to CSV

The eligibility windows are the shared ones in `../partners_lib.py`, the same the CNAF runs
on: AAH 16-30 ans, AEEH 6-19 ans.

Unlike the QF route, AAH and AEEH never need a quotient_familial call: MSA flags them
itself through `situation`, and the eligibility window only needs `date_naissance` -
both already on the phase-1 parquet. This notebook can therefore run as soon as
`clean_msa_1_before_qf_batch.ipynb` has finished, without waiting for qf-batch.ts.

⚠️ Run this notebook exactly once per campaign: `clean_msa_2_after_qf_batch.ipynb` (the QF
route) never recomputes AAH/AEEH, precisely so these beneficiaries are never handed a
second `id_psp` / a second campaign email.

In [ ]:
import csv
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# partners_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# partners_lib itself sits one level up, in partners/, next to the other partner folders.
partners_root = str(Path.cwd().parent)
if partners_root not in sys.path:
    sys.path.append(partners_root)

# Every step of this phase is shared with the other partners routed through qf-batch, so
# clean_msa_lib is only needed for the columns MSA drops on top of the shared list.
import partners_lib as partners
import clean_msa_lib as msa

load_dotenv()

base_output_filepath = os.environ['DB_MSA_EXPORT_2026_AAH_AEEH']

# Written by clean_msa_1_before_qf_batch.ipynb - keep both notebooks on the same value.
# It stays in this partner's own folder: unlike the qf-batch input/output, which live in the
# shared partners/qf-batch-workdir, this parquet is only ever read by the MSA notebooks.
msa_intermediate_filepath = os.environ.get(
    'MSA_INTERMEDIATE_PATHFILE_2026',
    str(Path.cwd() / 'msa_2026_pre_qf_batch.parquet'))

In [ ]:
# Cleaned, deduplicated beneficiary rows left by phase 1. They still carry the qf-batch
# pivot columns, dropped below - this route never needs qf-batch's output at all.
df_valid_no_duplicate = pd.read_parquet(msa_intermediate_filepath)

print(f"{len(df_valid_no_duplicate)} beneficiary row(s) read from {msa_intermediate_filepath}")

In [ ]:
## drop the columns now folded into the JSON columns, plus the qf-batch pivot-only ones
## (partners.FINAL_COLUMNS_TO_DROP, extended with what only MSA carries - see
## msa.MSA_EXTRA_COLUMNS_TO_DROP). `situation` and `date_naissance`, which the AAH/AEEH
## routes below need, are not part of either list and stay on df_final.
df_final = partners.drop_intermediate_columns(
    df_valid_no_duplicate, partners.FINAL_COLUMNS_TO_DROP + msa.MSA_EXTRA_COLUMNS_TO_DROP)

In [ ]:
# AAH route: 16-30 ans révolus - situation already settled from MSA's prestation.
# Computed directly on df_final: unlike the QF route, this never touches qf_value.
df_final_aah = partners.select_eligible_by_index(df_final, partners.aah_eligible_index(df_final))

# AEEH route: 6-19 ans révolus - no quotient_familial call needed, MSA already flags it
# (it spells the prestation "AEH")
df_final_aeeh = partners.select_eligible_by_index(df_final, partners.aeeh_eligible_index(df_final))

In [ ]:
# Merge AAH and AEEH routes
df_final_aah_and_aeeh = pd.concat(
    [df_final_aah, df_final_aeeh], ignore_index=True).reset_index(drop=True)

In [ ]:
# Cast to string
df_final_aah_and_aeeh.loc[:,'date_naissance'] = df_final_aah_and_aeeh['date_naissance'].astype(str)

In [ ]:
# output to CSV files
df_final_aah_and_aeeh.to_csv(base_output_filepath, sep=';', index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

In [ ]:
print(f"{len(df_final_aah)} df_final_aah")
print(f"{len(df_final_aeeh)} df_final_aeeh")
print(f"{len(df_final_aah_and_aeeh)} aah and aeeh written to {base_output_filepath}")

## ⏭ Next: generate_new_codes.ipynb
Run `generate_new_codes.ipynb` with `SOURCE = 'MSA_AAH_AEEH'` to assign these beneficiaries
their `id_psp`. Its dated `*-with-codes.csv` output is then what
`linkmobility/1_email_campaign.ipynb` reads (via `CAMPAIGN_INPUT_PATHFILE_2026`) to send
their campaign - independently of the QF route, which still waits on qf-batch.ts.